# Day 9 — FastAPI service

Wrap Day 8 predictor into `/health` and `/predict`.

In [ ]:
!pip -q install fastapi uvicorn python-multipart pandas numpy torch transformers pillow faiss-cpu

In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, uuid
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
print("BASE:", BASE)


In [ ]:

app_path = f"{BASE}/app.py"
app_code = r'''
import os, re, time, uuid, io
import numpy as np
import pandas as pd
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import faiss

BASE = os.environ.get("BASE", "__BASE__")
DATA_CSV = os.environ.get("DATA_CSV", os.path.join(BASE, "clean_dataset_2696.csv"))
FUSED_NPY = os.environ.get("FUSED_NPY", os.path.join(BASE, "fused_embeddings_alpha_0.5.npy"))
MIN_SCORE = float(os.environ.get("MIN_SCORE", "0.90"))
TOPK = int(os.environ.get("TOPK", "3"))

app = FastAPI(title="Radiology RAG Copilot", version="0.1.0")

df = pd.read_csv(DATA_CSV)
fused = np.load(FUSED_NPY).astype("float32")
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)
index = faiss.IndexFlatIP(fused.shape[1]); index.add(fused)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
model.eval()

def embed_pil(img: Image.Image) -> np.ndarray:
    inputs = processor(images=[img], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        v = model.get_image_features(**inputs)
        v = v / v.norm(dim=-1, keepdim=True)
    return v.cpu().numpy().astype("float32")[0]

def retrieve(v: np.ndarray, k=3):
    s, idx = index.search(v.reshape(1,-1).astype("float32"), k)
    cases=[]
    for j,(i,sc) in enumerate(zip(idx[0].tolist(), s[0].tolist()), start=1):
        cases.append({"row_idx":int(i),"study_id":int(df.loc[i,"study_id"]),
                      "score":float(sc),"impression":str(df.loc[i,"impression"]).strip()})
    best=float(s[0][0]) if len(s[0]) else 0.0
    return cases, best

def draft_from_cases(cases, k=3):
    bullets=[]
    for j,c in enumerate(cases[:k], start=1):
        txt=c["impression"].replace("\\n"," ").strip()
        first=re.split(r"(?<=[.!?])\\s+", txt)[0].strip()
        if first:
            bullets.append(f"- {first} [Case {j}]")
    return "IMPRESSION:\\n" + ("\\n".join(bullets) if bullets else "- Unable to draft.")

@app.get("/health")
def health():
    return {"status":"ok","device":device,"n_rows":len(df),"index_size":int(index.ntotal)}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    request_id = uuid.uuid4().hex[:8]
    t0=time.time()
    raw = await file.read()
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    v = embed_pil(img)
    cases,best = retrieve(v,k=TOPK)
    status = "generated" if best>=MIN_SCORE else "refused"
    draft = draft_from_cases(cases,k=TOPK)
    if status=="refused":
        draft="IMPRESSION:\\n- Unable to generate reliable draft due to low retrieval confidence."
    return JSONResponse({"request_id":request_id,"status":status,"confidence":float(best),
                         "latency_ms":int((time.time()-t0)*1000),"draft":draft,"retrieved_cases":cases})
'''
app_code = app_code.replace("__BASE__", BASE)
open(app_path,"w",encoding="utf-8").write(app_code)
print("Wrote:", app_path)


## Run server (keeps running)
Stop with the square stop button.

In [ ]:
!uvicorn app:app --host 0.0.0.0 --port 8000

## Test /health (NEW cell)

In [ ]:
import requests
r=requests.get('http://127.0.0.1:8000/health')
print(r.status_code, r.text)
